In [16]:
!pip install -U -q google-generativeai

In [17]:
import pandas as pd
from google import genai
from google.genai import types
import time
from tqdm import tqdm

In [26]:
model_name = 'models/gemini-flash-latest'

In [37]:
# 1. CONFIGURACIÓN PRO
# Usa tu API Key de Google AI Studio vinculada a tu cuenta Pro
from google.colab import userdata

API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)

In [28]:
# list(client.models.list())

In [29]:
# 2. CARGA DE DATOS
df = pd.read_csv('/content/sample_data/datos APPL.csv')

In [30]:
def analizar_lote(lista_tweets, retries=5):
    prompt = f"""
    Analiza el sentimiento de estos {len(lista_tweets)} tweets sobre Apple ($AAPL).
    Responde solo con una lista de etiquetas (Positivo, Negativo, Neutral) separadas por comas.
    Tweets:
    {chr(10).join([f"- {t}" for t in lista_tweets])}
    """

    for i in range(retries):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.1,  # Baja temperatura para respuestas consistentes
                )
            )
            return [s.strip().capitalize() for s in response.text.split(',')]
        except Exception as e:
            if "429" in str(e):
                time.sleep(2)
            else:
                print(f"Error inesperado: {e}")
                break
    return ["Error"] * len(lista_tweets)

In [31]:
# 4. EJECUCIÓN POR LOTES GRANDES (BATCH SIZE 50)
batch_size = 100
resultados_finales = []

print(f"Iniciando análisis de {len(df)} tweets con {model_name}...")

# Usamos tqdm para monitorizar el progreso en tiempo real
for i in tqdm(range(0, len(df), batch_size)):
    batch = df['Contenido_Texto'].iloc[i:i+batch_size].astype(str).tolist()

    # Llamada a la API
    resultados_batch = analizar_lote(batch)

    # Verificación de seguridad (que la IA devolvió el número correcto de etiquetas)
    if len(resultados_batch) != len(batch):
        # Si hay descuadre, reintentamos individualmente o marcamos error
        resultados_batch = resultados_batch[:len(batch)] + ["Error"] * (len(batch) - len(resultados_batch))

    resultados_finales.extend(resultados_batch)

    # Con cuenta Pro, un delay de 0.2s es suficiente para evitar límites
    time.sleep(10)


Iniciando análisis de 8181 tweets con models/gemini-flash-latest...


  6%|▌         | 5/82 [02:59<47:16, 36.83s/it]

Error inesperado: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


  9%|▊         | 7/82 [03:43<36:46, 29.42s/it]

Error inesperado: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


 18%|█▊        | 15/82 [08:03<37:57, 33.99s/it]

Error inesperado: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


 22%|██▏       | 18/82 [09:28<33:03, 31.00s/it]

Error inesperado: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


100%|██████████| 82/82 [31:50<00:00, 23.29s/it]


In [32]:
# 5. GUARDAR RESULTADOS
df['sentimiento_gemini'] = resultados_finales[:len(df)]
df.to_csv('tweets_analizados_gemini_pro.csv', index=False)

print("\n¡Proceso finalizado!")
print(df['sentimiento_gemini'].value_counts())


¡Proceso finalizado!
sentimiento_gemini
Error       6281
Positivo     753
Neutral      654
Negativo     363
Positive      98
Negative      32
Name: count, dtype: int64
